# Raha Detector

### Import Library

In [ ]:
import pickle
import yaml
import time
import raha

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

### Configuration

In [ ]:
# CONFIGURATION
RAHA_LABELING_BUDGET = 2
RAHA_VERBOSE = False
RAHA_AUTO_SAVE = False

# DICTIONARIES_YAML_PATH = "./dicts_minimal.yaml"
DICTIONARIES_YAML_PATH = "./dicts_full.yaml"
RAHA_DETECTION_RESULTS_PATH = "./results/detection/raha"

DetectionBaseline = raha.Detection()
DetectionBaseline.LABELING_BUDGET = RAHA_LABELING_BUDGET
DetectionBaseline.VERBOSE = RAHA_VERBOSE
DetectionBaseline.SAVE_RESULTS = RAHA_AUTO_SAVE

### Load YAML

In [ ]:
with open(DICTIONARIES_YAML_PATH, "rb") as file:
  dictionaries = yaml.safe_load(file)

### Detection

In [ ]:
for name, dct in dictionaries.items():
  start_time = time.time()
  print(f'Running detection on {name}')
  d = DetectionBaseline.initialize_dataset(dct)

  DetectionBaseline.run_strategies(d)
  DetectionBaseline.generate_features(d)
  DetectionBaseline.build_clusters(d)

  # auto sampling from ground truth
  while len(d.labeled_tuples) < DetectionBaseline.LABELING_BUDGET:
    DetectionBaseline.sample_tuple(d)
    if d.has_ground_truth:
      DetectionBaseline.label_with_ground_truth(d)
  
  DetectionBaseline.propagate_labels(d)
  DetectionBaseline.predict_labels(d) # result raha

  with open(f"{RAHA_DETECTION_RESULTS_PATH}/{name}.pkl", "wb") as f:
    pickle.dump(d, f)
    print(f'Pickle detection saved to {RAHA_DETECTION_RESULTS_PATH}/{name}.pkl')
    
  total_time = time.time() - start_time
  print(f"total time on {name}: {total_time:.1f} seconds")
  print(f"-" * 50)